<a href="https://colab.research.google.com/github/gurudattamanpreet/Practice/blob/main/pipeline_with_pca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score,classification_report
from sklearn.pipeline import Pipeline as SkPipeline
from imblearn.pipeline import Pipeline as ImbPipeline

In [3]:
url='https://github.com/gurudattamanpreet/datasets/raw/main/Tourism.xlsx'
df=pd.read_excel(url,sheet_name='Tourism')

In [4]:
df.shape

(4888, 20)

In [5]:
df.head()

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,200004,0,NaN,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0


In [6]:
df.drop(['CustomerID'],axis=1,inplace=True)

In [7]:
df.head()

,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0
3,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,0,NaN,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0


In [8]:
df.select_dtypes('object').nunique()

,0
TypeofContact,2
Occupation,4
Gender,3
ProductPitched,5
MaritalStatus,4
Designation,5


In [9]:
s=df.select_dtypes('object').columns.tolist()
for i in s:
  print(df[i].value_counts(),'\n')

TypeofContact
Self Enquiry       3444
Company Invited    1419
Name: count, dtype: int64 

Occupation
Salaried          2368
Small Business    2084
Large Business     434
Free Lancer          2
Name: count, dtype: int64 

Gender
Male       2916
Female     1817
Fe Male     155
Name: count, dtype: int64 

ProductPitched
Basic           1842
Deluxe          1732
Standard         742
Super Deluxe     342
King             230
Name: count, dtype: int64 

MaritalStatus
Married      2340
Divorced      950
Single        916
Unmarried     682
Name: count, dtype: int64 

Designation
Executive         1842
Manager           1732
Senior Manager     742
AVP                342
VP                 230
Name: count, dtype: int64 



In [10]:
df['Gender']=df['Gender'].replace('Fe Male','Female')

In [11]:
df['ProdTaken'].isna().sum()

np.int64(0)

In [12]:
X_train,X_test,y_train,y_test=train_test_split(df.drop(['ProdTaken'],axis=1),df['ProdTaken'],test_size=0.2,random_state=1,stratify=df['ProdTaken'])

In [13]:
X_train.shape

(3910, 18)

In [14]:
X_test.shape

(978, 18)

In [15]:
num_cols=X_train.select_dtypes('number').columns.tolist()
cat_cols=X_train.select_dtypes('object').columns.tolist()

num_pipeline=SkPipeline([('impute',SimpleImputer(strategy='median')),('scaler',StandardScaler())])
cat_pipeline=SkPipeline([('impute',SimpleImputer(strategy='most_frequent')),('encode',OneHotEncoder(drop='first',handle_unknown='ignore',sparse_output=False))])

preprocessor=ColumnTransformer([('num',num_pipeline,num_cols),('cat',cat_pipeline,cat_cols)],remainder='passthrough')

pca = PCA(n_components=10)

base_models = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
    ('dt', DecisionTreeClassifier(random_state=42))
]

# Final model (meta-learner)
final_model = LogisticRegression()

# Stacking classifier
stack_model = StackingClassifier(
    estimators=base_models,
    final_estimator=final_model,
    passthrough=False,
    cv=5
)

pipe=ImbPipeline([
    ('preprocessor',preprocessor),
    ('pca',pca),
    ('smote',SMOTE()),
    ('model', stack_model)])

In [16]:
# grid_search=GridSearchCV(pipe,param_grid,cv=5,scoring='accuracy',verbose=2,n_jobs=-1)

In [17]:
pipe.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'CityTier',
                                                   'DurationOfPitch',
                                                   'NumberOfPersonVisiting',
                                                   'NumberOfFollowups',
                                                   'PreferredPropertyStar',
                                                   'NumberOfTrips', 'Passport',
                                                   'PitchSatisfactionScore',
                                                   'OwnCar'...
                                                   'Occupation', 'Gender',
                                                   'ProductPitched',
                                                   'MaritalStatus',
                                                   'Designation'])])),
                ('pca', PCA(n_components=10)), ('smote', SMOTE()),
                ('model',
                 StackingClassifier(cv=5,
                                    estimators=[('rf',
                                                 RandomForestClassifier(random_state=42)),
                                                ('gb',
                                                 GradientBoostingClassifier(random_state=42)),
                                                ('dt',
                                                 DecisionTreeClassifier(random_state=42))],
                                    final_estimator=LogisticRegression()))])

In [18]:
y_pred_train=pipe.predict(X_train)
y_pred_test=pipe.predict(X_test)

In [19]:
accuracy_score(y_test,y_pred_test)

0.9171779141104295

In [20]:
accuracy_score(y_train,y_pred_train)

1.0

    PCA kehta hai: "Main tumhe best angle batata hun jahan se dekhoge to maximum information milegi!"
    
    Bohot saare features (100+ columns) ko
    Kam important features (2-3 columns) mein convert karna
    Maximum information retain rakhte hue

    Real Life Use:

    Instagram Photos: 1000+ pixels → 50 important features
    Stock Market: 500+ indicators → 3-4 main trends
    Face Recognition: 10000+ pixel values → 100 key features

    PCA bas itna karta hai:

    “Main ye dekh raha hoon ki height aur weight ka combination data ke
    variation ko best explain karta hai,
    to main un dono ko mila ke ek new axis (PC1) bana deta hoon.”

In [ ]:
# pca=PCA(n_components=50)
# pca.fit()

In [ ]:
pca_model = grid_search.best_estimator_.named_steps['pca']
print("Explained variance ratio:", pca_model.explained_variance_ratio_)
print("Total variance retained:", pca_model.explained_variance_ratio_.sum())

In [ ]:
explained_var = pca_model.explained_variance_ratio_

plt.figure(figsize=(8,5))
plt.plot(np.cumsum(explained_var), marker='o')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA - Cumulative Explained Variance')
plt.grid(True)
plt.show()

              Input Features
                  |
          -------------------------
          |           |           |
      Decision   Random       XGBoost
        Tree       Forest      Classifier
          |           |           |
          ------------+-----------+---------→ predictions
                                |
                        Meta-model (Logistic Regression)
                                |
                          Final Prediction ✅

In [34]:
a = input("Enter a word: ")

vowel='aeiou'
if a.isalpha():
  print(sum(1 for x in a if x in vowel))
else:
  print("Enter valid string")

Enter a word: 6372
Enter valid string


galat h
